# 📦 Virtual Environments & Python Packaging

> **What you'll learn:** Why virtual environments exist, how to use venv/conda/poetry, pip dependency management, creating your own Python package, and understanding pyproject.toml.

---

**Difficulty:** Beginner | **Time:** 2–3 hours | **Prerequisites:** Python Basics

## 🤔 The Problem — Why Do We Need This?

Imagine you have two projects on your computer:
- **Project A** needs `numpy==1.20` (an old version)
- **Project B** needs `numpy==1.26` (a new version)

If you install numpy globally on your system, you can only have **one version**. Installing 1.26 for Project B breaks Project A. This is called **dependency conflict**.

**Virtual environments** solve this by giving each project its **own isolated copy** of Python and its packages.

**Analogy:** Think of each project as a chef with their own kitchen. Each kitchen has exactly the ingredients they need, and they don't interfere with each other.

```
Without virtual envs:          With virtual envs:

[System Python]                [Project A env]   [Project B env]
numpy 1.20 (or 1.26?)          numpy 1.20         numpy 1.26
This conflicts! ❌              Works! ✅           Works! ✅
```

## 📋 Table of Contents

1. [venv — Python's Built-in Solution](#1)
2. [pip — The Package Installer](#2)
3. [requirements.txt](#3)
4. [conda — The Data Science Choice](#4)
5. [pipenv and poetry — Modern Alternatives](#5)
6. [Python Modules and Packages](#6)
7. [Creating Your Own Package](#7)
8. [pyproject.toml and setup.py](#8)
9. [.gitignore for Python](#9)
10. [Mini Project](#10)
11. [Interview Q&A](#11)
12. [Resources](#12)

## 1. venv — Python's Built-in Solution <a id='1'></a>

`venv` is built into Python (no install needed). It creates an isolated environment in a folder.

**All commands below are run in your terminal, not in Python.**

In [ ]:
# This cell shows the terminal commands you'd run
# They won't run in a notebook — they go in your terminal/shell

commands = """
# ============================================================
# CREATING AND USING VIRTUAL ENVIRONMENTS
# ============================================================

# 1. Create a virtual environment named 'venv' (or any name)
python -m venv venv

# Common names: venv, .venv, env, .env
# Many people use .venv (hidden on Unix with the dot)
python -m venv .venv

# 2. ACTIVATE the environment
# On Mac/Linux:
source .venv/bin/activate
# On Windows (Command Prompt):
.venv\\Scripts\\activate.bat
# On Windows (PowerShell):
.venv\\Scripts\\Activate.ps1

# After activation, your prompt changes:
# (.venv) $  ← you see the env name in parentheses

# 3. Verify you're using the right Python:
which python         # Mac/Linux: should show path inside .venv
where python         # Windows: should show path inside .venv
python --version     # check version

# 4. Install packages INTO the virtual environment:
pip install numpy
pip install pandas matplotlib scikit-learn
pip install requests==2.31.0  # specific version

# 5. DEACTIVATE when done:
deactivate
# Your prompt goes back to normal

# 6. Reactivate next time:
source .venv/bin/activate   # same as step 2
"""

print(commands)

In [ ]:
# We CAN demonstrate the concept in Python:
import sys
import os

# Check what Python interpreter is currently running:
print("Python executable:", sys.executable)
print("Python version:", sys.version[:20])

# Check if we're inside a virtual environment:
def in_virtualenv():
    return (
        hasattr(sys, 'real_prefix') or          # old virtualenv
        (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix)  # venv
    )

print("In virtual environment:", in_virtualenv())
print("sys.prefix:", sys.prefix[:50])

## 2. pip — The Package Installer <a id='2'></a>

pip is Python's package manager. It downloads and installs packages from [PyPI](https://pypi.org) (Python Package Index) — a repository of 500,000+ free packages.

In [ ]:
pip_commands = """
# ============================================================
# PIP COMMANDS (run in terminal with venv activated)
# ============================================================

# Install a package:
pip install numpy

# Install a specific version:
pip install numpy==1.24.0

# Install minimum version:
pip install numpy>=1.24

# Install a range of versions:
pip install 'numpy>=1.24,<2.0'

# Install multiple packages at once:
pip install numpy pandas matplotlib scikit-learn

# Upgrade a package:
pip install --upgrade numpy

# Uninstall a package:
pip uninstall numpy
pip uninstall numpy -y    # skip confirmation prompt

# List installed packages:
pip list

# Show info about a package:
pip show numpy
# Shows: Name, Version, Location, Requires, etc.

# Search PyPI (deprecated, use pypi.org website instead):
# pip search numpy  # no longer works

# Check for outdated packages:
pip list --outdated

# Install from a requirements file:
pip install -r requirements.txt

# Install current project in editable mode (for development):
pip install -e .
"""

print(pip_commands)

In [ ]:
# Demonstrate: checking installed packages programmatically
import importlib.metadata as importlib_metadata

def get_package_version(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return "not installed"

# Check which ML packages are available:
packages_to_check = [
    "numpy", "pandas", "matplotlib", "scikit-learn",
    "torch", "tensorflow", "xgboost", "lightgbm"
]

print("Package versions:")
for pkg in packages_to_check:
    version = get_package_version(pkg)
    status = "✅" if version != "not installed" else "❌"
    print(f"  {status} {pkg}: {version}")

## 3. requirements.txt <a id='3'></a>

A `requirements.txt` file lists all packages your project needs. Anyone can recreate your environment with one command.

**Analogy:** It's like a recipe — anyone who has the recipe can cook the same dish (reproduce your environment).

In [ ]:
# Show what a requirements.txt looks like:
requirements_example = """# requirements.txt — project dependencies

# Exact versions (most reproducible):
numpy==1.26.2
pandas==2.1.3
matplotlib==3.8.1

# Minimum versions (more flexible):
scikit-learn>=1.3.0
requests>=2.28.0

# Version range:
fastapi>=0.100.0,<1.0.0

# Install from git:
# git+https://github.com/user/repo.git
"""

print(requirements_example)

# Terminal commands for requirements.txt:
commands = """
# Create requirements.txt from current environment:
pip freeze > requirements.txt

# Install from requirements.txt:
pip install -r requirements.txt

# Better: separate dev requirements
# requirements.txt          ← production dependencies only
# requirements-dev.txt      ← + testing, linting, formatting
pip install -r requirements-dev.txt
"""
print(commands)

# Demonstrate: generating requirements content (read-only)
import subprocess
import sys

try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "list", "--format=freeze"],
        capture_output=True, text=True
    )
    lines = result.stdout.strip().split("\n")[:10]  # first 10 packages
    print("First 10 installed packages:")
    for line in lines:
        print(f"  {line}")
except Exception as e:
    print(f"Could not run pip: {e}")

## 4. conda — The Data Science Choice <a id='4'></a>

conda is an alternative to venv + pip. It's popular in data science because it:
- Manages Python itself (not just packages)
- Handles non-Python dependencies (C libraries, CUDA, etc.)
- Comes with Anaconda/Miniconda

In [ ]:
conda_commands = """
# ============================================================
# CONDA COMMANDS (after installing Anaconda or Miniconda)
# ============================================================

# Create environment with specific Python version:
conda create -n myenv python=3.11
conda create -n mlenv python=3.10 numpy pandas scikit-learn

# Activate:
conda activate myenv

# Deactivate:
conda deactivate

# Install packages:
conda install numpy
conda install -c conda-forge lightgbm    # -c = channel

# Some packages must use pip even in conda:
pip install some_package

# List all environments:
conda env list
# Or:
conda info --envs

# List packages in current env:
conda list

# Export environment to file:
conda env export > environment.yml

# Create env from file:
conda env create -f environment.yml

# Delete an environment:
conda env remove -n myenv

# Update conda itself:
conda update conda
"""
print(conda_commands)

# What environment.yml looks like:
environment_yml = """
# environment.yml
name: mlenv
channels:
  - defaults
  - conda-forge
dependencies:
  - python=3.10
  - numpy=1.26
  - pandas=2.1
  - scikit-learn=1.3
  - pip
  - pip:
    - some-pypi-only-package==1.0
"""
print("environment.yml example:")
print(environment_yml)

## 5. pipenv and poetry — Modern Alternatives <a id='5'></a>

**poetry** is the most modern Python dependency manager. It handles venv creation, dependency resolution, and publishing — all in one tool.

**When to use which:**
- `venv + pip` → simple projects, maximum compatibility
- `conda` → data science, GPU/CUDA dependencies  
- `poetry` → professional projects, library publishing

In [ ]:
poetry_intro = """
# ============================================================
# POETRY (install: pip install poetry  OR  pipx install poetry)
# ============================================================

# Create a new project:
poetry new my-project
# Creates:
# my-project/
#   pyproject.toml      ← modern config file
#   README.md
#   my_project/
#     __init__.py
#   tests/
#     test_my_project.py

# Or start in existing directory:
poetry init

# Add dependencies:
poetry add numpy pandas
poetry add pytest --group dev    # dev-only dependency

# Remove:
poetry remove numpy

# Install all dependencies:
poetry install

# Run a command in the virtual environment:
poetry run python main.py
poetry run pytest

# Activate the shell:
poetry shell

# Show installed packages:
poetry show
poetry show --tree    # show dependency tree

# Update dependencies:
poetry update

# Publish to PyPI:
poetry publish --build
"""
print(poetry_intro)

## 6. Python Modules and Packages <a id='6'></a>

- A **module** is a single `.py` file
- A **package** is a folder containing an `__init__.py` file (and usually multiple modules)

In [ ]:
# Understanding Python's import system
import sys
import os

# When you write: import numpy
# Python searches these locations in order:
print("Python looks for modules in:")
for path in sys.path:
    if path:  # skip empty string
        print(f"  {path}")

# Demonstrate: creating a simple module in memory
# (In reality, you'd create a file like math_utils.py)

math_utils_content = '''
"""math_utils.py — a simple math utility module."""

PI = 3.14159265358979

def circle_area(radius):
    """Calculate area of a circle."""
    return PI * radius ** 2

def circle_circumference(radius):
    """Calculate circumference of a circle."""
    return 2 * PI * radius

def is_prime(n):
    """Check if n is a prime number."""
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

if __name__ == "__main__":
    # This only runs when the file is executed directly
    # NOT when it's imported
    print("Running math_utils.py directly")
    print(f"Area of circle r=5: {circle_area(5):.2f}")
'''

# Write the module file
with open("math_utils.py", "w") as f:
    f.write(math_utils_content)

# Now import it:
import math_utils

print(f"PI = {math_utils.PI}")
print(f"Area of r=5: {math_utils.circle_area(5):.2f}")
print(f"Is 17 prime? {math_utils.is_prime(17)}")
print(f"Is 18 prime? {math_utils.is_prime(18)}")

# Different import styles:
from math_utils import circle_area, is_prime
print(f"Direct import: {circle_area(3):.2f}")

# Cleanup
os.remove("math_utils.py")

## 7. Creating Your Own Package <a id='7'></a>

A package is a folder with `__init__.py`. You can import from it just like any installed library.

In [ ]:
import os

# Create a simple package structure:
# mypackage/
#   __init__.py     <- makes it a package
#   math_ops.py
#   text_ops.py

os.makedirs("mypackage", exist_ok=True)

# __init__.py — runs when the package is imported
with open("mypackage/__init__.py", "w") as f:
    f.write('''
"""mypackage — a sample Python package."""

__version__ = "1.0.0"
__author__ = "Alice"

# Import commonly used things so users can do:
# from mypackage import circle_area
from .math_ops import circle_area, is_prime
from .text_ops import count_words, reverse_words
''')

# math_ops.py
with open("mypackage/math_ops.py", "w") as f:
    f.write('''
import math

def circle_area(radius):
    return math.pi * radius ** 2

def is_prime(n):
    if n < 2: return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0: return False
    return True

def factorial(n):
    if n == 0: return 1
    return n * factorial(n - 1)
''')

# text_ops.py
with open("mypackage/text_ops.py", "w") as f:
    f.write('''
def count_words(text):
    return len(text.split())

def reverse_words(text):
    return " ".join(text.split()[::-1])

def title_case(text):
    return " ".join(w.capitalize() for w in text.split())
''')

# Now use the package!
import sys
sys.path.insert(0, ".")  # add current dir to path
import importlib

import mypackage
print(f"Package: {mypackage.__version__} by {mypackage.__author__}")
print(f"circle_area(5) = {mypackage.circle_area(5):.2f}")
print(f"is_prime(13) = {mypackage.is_prime(13)}")
print(f"count_words('hello world') = {mypackage.count_words('hello world')}")
print(f"reverse_words('I love Python') = '{mypackage.reverse_words('I love Python')}'")

# Direct submodule import:
from mypackage.math_ops import factorial
print(f"factorial(10) = {factorial(10)}")

# Cleanup
import shutil
shutil.rmtree("mypackage")

## 8. pyproject.toml and setup.py <a id='8'></a>

`pyproject.toml` is the modern standard for configuring Python projects (PEP 518). It replaces `setup.py` and `setup.cfg`.

In [ ]:
pyproject_example = """
# pyproject.toml — modern Python project configuration

[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "my-awesome-package"
version = "1.0.0"
description = "A short description of what my package does"
readme = "README.md"
requires-python = ">=3.9"
license = {text = "MIT"}
authors = [
    {name = "Alice Smith", email = "alice@example.com"}
]
keywords = ["python", "machine-learning", "awesome"]
classifiers = [
    "Development Status :: 5 - Production/Stable",
    "Programming Language :: Python :: 3",
    "License :: OSI Approved :: MIT License",
]

# Runtime dependencies (required to use the package):
dependencies = [
    "numpy>=1.24",
    "pandas>=2.0",
    "requests>=2.28",
]

[project.optional-dependencies]
dev = ["pytest", "black", "ruff", "mypy"]
docs = ["sphinx", "sphinx-rtd-theme"]

[project.urls]
Homepage = "https://github.com/alice/my-awesome-package"
Documentation = "https://my-awesome-package.readthedocs.io"
Bug-Tracker = "https://github.com/alice/my-awesome-package/issues"

[project.scripts]
# Creates a command-line tool 'my-tool' when installed
my-tool = "my_awesome_package.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.black]
line-length = 88
target-version = ['py39']

[tool.ruff]
line-length = 88
select = ["E", "F", "I"]
"""
print(pyproject_example)

In [ ]:
# Publishing to PyPI — the basic steps
publishing_steps = """
# Publishing your package to PyPI (https://pypi.org)

# 1. Create accounts:
#    - https://pypi.org (production)
#    - https://test.pypi.org (for testing)

# 2. Install build tools:
pip install build twine

# 3. Build your package:
python -m build
# Creates: dist/
#   my-awesome-package-1.0.0.tar.gz
#   my-awesome-package-1.0.0-py3-none-any.whl

# 4. Upload to TestPyPI first (safe!):
python -m twine upload --repository testpypi dist/*
# Test install:
pip install --index-url https://test.pypi.org/simple/ my-awesome-package

# 5. Upload to real PyPI:
python -m twine upload dist/*

# 6. Now anyone can install:
pip install my-awesome-package

# With poetry it's even simpler:
poetry publish --build
"""
print(publishing_steps)

## 9. .gitignore for Python <a id='9'></a>

Never commit your virtual environment or compiled files to git. Always add a `.gitignore`.

In [ ]:
gitignore_content = """# Python .gitignore — add this to every Python project!

# Virtual environments (NEVER commit these)
venv/
.venv/
env/
.env/
ENV/

# Python cache files
__pycache__/
*.py[cod]
*$py.class
*.pyc

# Distribution / packaging
dist/
build/
*.egg-info/
*.egg

# Testing
.pytest_cache/
.coverage
coverage.xml
htmlcov/
.tox/

# Type checking
.mypy_cache/
.ruff_cache/

# Jupyter Notebooks
.ipynb_checkpoints/

# Environment variables (NEVER commit secrets!)
.env
*.env

# IDE files
.vscode/
.idea/
*.swp
*.swo

# macOS
.DS_Store

# Windows
Thumbs.db
ehthumbs.db

# Data files (usually too large for git)
*.csv
*.xlsx
data/*.parquet
models/*.pkl
"""

print(gitignore_content)

## 10. 🚀 Mini Project: Setting Up a Real ML Project <a id='10'></a>

Let's set up a complete, professional Python project structure.

In [ ]:
import os
import json

# Simulate creating a professional ML project structure
project_name = "churn_predictor"

structure = {
    f"{project_name}/": {
        "README.md": "# Churn Predictor\n\nPredicts customer churn using ML.",
        "pyproject.toml": """[project]\nname = \"churn-predictor\"\nversion = \"0.1.0\"\n""",
        ".gitignore": "venv/\n__pycache__/\n*.pyc\n.env\n",
        "requirements.txt": "numpy>=1.24\npandas>=2.0\nscikit-learn>=1.3\nxgboost>=2.0\nmlflow>=2.8\n",
        "requirements-dev.txt": "-r requirements.txt\npytest\nblack\nruff\njupyter\n",
        ".env.example": "MLFLOW_TRACKING_URI=http://localhost:5000\nDATA_PATH=data/raw/\n",
        "src/churn_predictor/__init__.py": "__version__ = '0.1.0'\n",
        "src/churn_predictor/data.py": "# Data loading and preprocessing\n",
        "src/churn_predictor/features.py": "# Feature engineering\n",
        "src/churn_predictor/model.py": "# Model training and evaluation\n",
        "src/churn_predictor/predict.py": "# Inference code\n",
        "tests/__init__.py": "",
        "tests/test_data.py": "# Tests for data.py\n",
        "tests/test_model.py": "# Tests for model.py\n",
        "notebooks/01_EDA.ipynb": "",
        "notebooks/02_Feature_Engineering.ipynb": "",
        "data/raw/.gitkeep": "",
        "data/processed/.gitkeep": "",
        "models/.gitkeep": "",
    }
}

def display_tree(structure, prefix=""):
    """Display a project structure as a tree."""
    items = list(structure.items())
    for i, (path, content) in enumerate(items):
        is_last = i == len(items) - 1
        connector = "└── " if is_last else "├── "
        print(prefix + connector + path)

# Print as tree
print(f"📁 {project_name}/")
for path in sorted(structure[f"{project_name}/"].keys()):
    parts = path.split("/")
    indent = "│   " * (len(parts) - 1)
    print(f"   {indent}├── {parts[-1]}")

print("\n=" * 50)
print("Project structure explained:")
print("")
print("src/churn_predictor/  ← your actual code package")
print("tests/                ← unit tests (run with pytest)")
print("notebooks/            ← Jupyter notebooks for exploration")
print("data/                 ← data files (not committed to git)")
print("models/               ← saved model files")
print("requirements.txt      ← production dependencies")
print("requirements-dev.txt  ← dev/test dependencies")
print(".env.example          ← template for secrets (commit this)")
print(".gitignore            ← what git should ignore")

## 11. 🎤 Interview Questions & Answers <a id='11'></a>

**Q1: Why do we use virtual environments?**  
**A:** To isolate project dependencies. Each project gets its own Python environment with its own package versions, preventing conflicts between projects.

**Q2: What is the difference between `venv` and `conda`?**  
**A:** `venv` only manages Python packages. `conda` manages Python itself, packages, and non-Python dependencies (C libraries, CUDA). Conda is preferred for data science due to complex C/CUDA dependencies; venv is simpler for general Python projects.

**Q3: What is the difference between `pip freeze` and `pip list`?**  
**A:** `pip list` shows packages in a human-readable table. `pip freeze` shows them in `requirements.txt` format (`package==version`), suitable for saving to a file.

**Q4: What is `pip install -e .` and when would you use it?**  
**A:** Editable install — installs your local package so changes to the source code are immediately reflected without reinstalling. Used when developing a package and testing it simultaneously.

**Q5: What is `__init__.py` and why is it needed?**  
**A:** It marks a directory as a Python package. Without it, Python won't recognize the directory as importable. It can be empty or contain initialization code and exports.

**Q6: What is the difference between `__init__.py` and `pyproject.toml`?**  
**A:** `__init__.py` makes a directory a Python package (importable). `pyproject.toml` is a project configuration file — it defines metadata, dependencies, and build settings for distributing your project.

**Q7: How do you handle secrets (API keys) in a Python project?**  
**A:** Store them in a `.env` file (never commit to git). Load them with the `python-dotenv` library: `load_dotenv()` then `os.getenv("API_KEY")`. Always add `.env` to `.gitignore` and provide a `.env.example` template.

**Q8: What is the difference between `poetry` and `pip + venv`?**  
**A:** Poetry combines virtual environment creation, dependency resolution, version locking, and package publishing into one tool. pip + venv requires manual orchestration of these steps. Poetry is better for team projects and publishing libraries.

## 12. 📚 Resources & Further Learning <a id='12'></a>

### Official Docs
- [Python venv](https://docs.python.org/3/library/venv.html)
- [pip documentation](https://pip.pypa.io/en/stable/)
- [PyPI (Python Package Index)](https://pypi.org)
- [Poetry documentation](https://python-poetry.org/docs/)

### YouTube
- [Corey Schafer — Virtual Environments](https://www.youtube.com/watch?v=Kg1Yvry_Ydk)
- [Real Python — Virtual Environments](https://www.youtube.com/watch?v=APOPm01BVrk)
- [ArjanCodes — Python Project Setup](https://www.youtube.com/watch?v=5KEObONUkik)

### Guides
- [Real Python — Virtual Environments Primer](https://realpython.com/python-virtual-environments-a-primer/)
- [Python Packaging User Guide](https://packaging.python.org/en/latest/)

## ✅ Summary

| Tool | Purpose |
|------|---------|
| `venv` | Create isolated Python environments |
| `pip` | Install/manage packages from PyPI |
| `requirements.txt` | Reproducible dependency list |
| `conda` | Manage Python + packages + C deps |
| `poetry` | Modern all-in-one project management |
| `__init__.py` | Makes a folder a Python package |
| `pyproject.toml` | Modern project configuration |
| `.gitignore` | Keep venv/secrets out of git |

### The Golden Rule
> **Always use a virtual environment. Never install packages globally.**

```
# Your workflow for every new project:
mkdir my_project && cd my_project
python -m venv .venv
source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install -r requirements.txt
```

## ➡️ What's Next?

You've completed the Foundations! Open `01_Core_Scientific_Computing/NumPy.ipynb` to start learning NumPy — the backbone of all numerical computing in Python.